# EDA: Анализ данных Car Parts and Damage Dataset

**Цель:** Провести первичный анализ датасета для задачи instance segmentation повреждений и деталей автомобиля.

**Этапы:**
1. Загрузка и обзор структуры данных
2. Анализ распределения классов
3. Анализ размеров изображений
4. Анализ аннотаций (полигоны, площади, количество объектов на изображение)
5. Визуализация примеров с аннотациями
6. Выводы и рекомендации

## 1. Загрузка и обзор структуры данных

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from collections import Counter
import cv2
from PIL import Image

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

from car_damage_segmentation.utils import load_json, save_json
from car_damage_segmentation.data import polygon_to_mask

sns.set_theme(style='whitegrid', palette='muted', font='Segoe UI')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'Segoe UI'

print('Библиотеки загружены.')

In [ ]:
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

all_records = load_json(PROCESSED_DIR / 'all_records.json')
labels = load_json(PROCESSED_DIR / 'labels.json')
train_records = load_json(PROCESSED_DIR / 'train_records.json')
val_records = load_json(PROCESSED_DIR / 'val_records.json')

class_names = [c['name'] for c in labels['classes']]
class_categories = {c['name']: c['supercategory'] for c in labels['classes']}

print(f'Всего изображений: {len(all_records)}')
print(f'Train: {len(train_records)}')
print(f'Val:   {len(val_records)}')
print(f'Классов: {len(class_names)}')
print(f'\nКлассы: {", ".join(class_names)}')

## 2. Анализ распределения классов

In [ ]:
class_counter = Counter()
supercategory_counter = Counter()

for record in all_records:
    for ann in record['annotations']:
        cat_name = ann['category_name']
        class_counter[cat_name] += 1
        supercat = class_categories.get(cat_name, 'unknown')
        supercategory_counter[supercat] += 1

df_class = pd.DataFrame(
    sorted(class_counter.items(), key=lambda x: -x[1]),
    columns=['Класс', 'Количество']
)
df_class['%'] = (df_class['Количество'] / df_class['Количество'].sum() * 100).round(2)
df_class['Тип'] = df_class['Класс'].map(lambda c: class_categories.get(c, 'unknown'))
df_class

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 10))

colors = ['#e74c3c' if t == 'damage' else '#3498db' for t in df_class['Тип']]
axes[0].barh(df_class['Класс'], df_class['Количество'], color=colors)
axes[0].set_xlabel('Количество экземпляров')
axes[0].set_title('Распределение классов (детали vs повреждения)')
axes[0].invert_yaxis()

for i, (val, pct) in enumerate(zip(df_class['Количество'], df_class['%'])):
    axes[0].text(val + 5, i, f'{val} ({pct}%)', va='center', fontsize=7)

sc_labels = list(supercategory_counter.keys())
sc_values = list(supercategory_counter.values())
axes[1].pie(sc_values, labels=sc_labels, autopct='%1.1f%%', 
            colors=['#e74c3c', '#3498db'], startangle=90,
            explode=(0.03, 0))
axes[1].set_title('Соотношение повреждений и деталей')

plt.tight_layout()
plt.show()

**Вывод:** Сильный дисбаланс классов — деталей значительно больше, чем повреждений. Это ожидаемо для задачи автострахования. При обучении стоит учитывать class weights или аугментации редких классов.

## 3. Анализ размеров изображений

In [ ]:
widths, heights, ratios = [], [], []
missing_images = []

for record in all_records:
    w, h = record['width'], record['height']
    widths.append(w)
    heights.append(h)
    ratios.append(w / h)

widths = np.array(widths)
heights = np.array(heights)
ratios = np.array(ratios)

print(f'Ширина:  min={widths.min():.0f}, max={widths.max():.0f}, mean={widths.mean():.0f}, median={np.median(widths):.0f}')
print(f'Высота:  min={heights.min():.0f}, max={heights.max():.0f}, mean={heights.mean():.0f}, median={np.median(heights):.0f}')
print(f'Aspect ratio: min={ratios.min():.2f}, max={ratios.max():.2f}, mean={ratios.mean():.2f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(widths, bins=30, color='#3498db', edgecolor='white', alpha=0.8)
axes[0].axvline(np.median(widths), color='red', linestyle='--', label=f'Median: {np.median(widths):.0f}')
axes[0].set_xlabel('Ширина (px)')
axes[0].set_ylabel('Количество')
axes[0].set_title('Распределение ширины изображений')
axes[0].legend()

axes[1].hist(heights, bins=30, color='#2ecc71', edgecolor='white', alpha=0.8)
axes[1].axvline(np.median(heights), color='red', linestyle='--', label=f'Median: {np.median(heights):.0f}')
axes[1].set_xlabel('Высота (px)')
axes[1].set_title('Распределение высоты изображений')
axes[1].legend()

axes[2].scatter(widths, heights, alpha=0.4, s=8, c='#9b59b6')
axes[2].set_xlabel('Ширина (px)')
axes[2].set_ylabel('Высота (px)')
axes[2].set_title('Ширина vs Высота')

plt.tight_layout()
plt.show()

**Вывод:** Изображения имеют разнообразные размеры. Потребуется resize/падинг перед подачей в модель. Бэкбон Mask R-CNN ожидает фиксированный вход.

## 4. Анализ аннотаций

In [ ]:
objects_per_image = []
polygon_points = []
mask_areas = []

for record in all_records:
    n_objects = len(record['annotations'])
    objects_per_image.append(n_objects)
    
    for ann in record['annotations']:
        n_points = len(ann['polygon']) // 2
        polygon_points.append(n_points)
        
        mask = polygon_to_mask(
            height=record['height'],
            width=record['width'],
            polygon=ann['polygon'],
            holes=ann.get('holes', [])
        )
        mask_areas.append(int(mask.sum()))

objects_per_image = np.array(objects_per_image)
polygon_points = np.array(polygon_points)
mask_areas = np.array(mask_areas)

print(f'Объектов на изображение: min={objects_per_image.min()}, max={objects_per_image.max()}, mean={objects_per_image.mean():.1f}, median={np.median(objects_per_image):.0f}')
print(f'Точек в полигоне: min={polygon_points.min()}, max={polygon_points.max()}, mean={polygon_points.mean():.1f}')
print(f'Площадь маски (px): min={mask_areas.min()}, max={mask_areas.max()}, mean={mask_areas.mean():.0f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].hist(objects_per_image, bins=range(0, objects_per_image.max() + 2), 
                color='#e67e22', edgecolor='white', alpha=0.8)
axes[0, 0].axvline(np.median(objects_per_image), color='red', linestyle='--', 
                   label=f'Median: {np.median(objects_per_image):.0f}')
axes[0, 0].set_xlabel('Количество объектов')
axes[0, 0].set_title('Объектов на изображение')
axes[0, 0].legend()

axes[0, 1].hist(polygon_points, bins=40, color='#8e44ad', edgecolor='white', alpha=0.8)
axes[0, 1].set_xlabel('Количество точек в полигоне')
axes[0, 1].set_title('Сложность полигонов')

axes[1, 0].hist(mask_areas, bins=50, color='#16a085', edgecolor='white', alpha=0.8)
axes[1, 0].set_xlabel('Площадь маски (px)')
axes[1, 0].set_title('Распределение площади масок')
axes[1, 0].set_yscale('log')

small_masks = np.sum(mask_areas < 100)
total_masks = len(mask_areas)
axes[1, 1].text(0.5, 0.5, 
                f'Масок всего: {total_masks:,}\n'
                f'Мелких (<100 px): {small_masks:,} ({small_masks/total_masks*100:.1f}%)\n'
                f'Средняя площадь: {mask_areas.mean():.0f} px\n'
                f'Медианная площадь: {np.median(mask_areas):.0f} px',
                ha='center', va='center', fontsize=14,
                bbox=dict(boxstyle='round', facecolor='#ecf0f1'))
axes[1, 1].set_title('Сводка по маскам')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

**Вывод:** Большинство изображений содержит 1-5 объектов. Есть значительное количество мелких масок (<100 px), что может затруднить детекцию. Полигоны варьируются по сложности от простых (4 точки) до детальных (>50 точек).

## 5. Визуализация примеров с аннотациями

In [ ]:
def visualize_sample(record, ax, title=None):
    img_path = record['image_path']
    img = cv2.imread(img_path)
    if img is None:
        ax.text(0.5, 0.5, f'Image not found:\n{img_path}', ha='center', va='center')
        return
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    overlay = img.copy()
    for ann in record['annotations']:
        mask = polygon_to_mask(
            height=record['height'],
            width=record['width'],
            polygon=ann['polygon'],
            holes=ann.get('holes', [])
        )
        color = np.random.randint(0, 255, 3).tolist()
        overlay[mask > 0] = (overlay[mask > 0] * 0.55 + np.array(color) * 0.45).astype(np.uint8)
        
        contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(overlay, contours, -1, color, 2)
        
        ys, xs = np.where(mask > 0)
        if len(xs) > 0:
            cx, cy = xs.mean(), ys.mean()
            cv2.putText(overlay, ann['category_name'][:12], (int(cx) - 20, int(cy)),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1)
    
    ax.imshow(overlay)
    ax.set_title(title or f"{Path(img_path).name}\n{len(record['annotations'])} объектов")
    ax.axis('off')


np.random.seed(42)
sample_indices = np.random.choice(len(all_records), min(8, len(all_records)), replace=False)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for idx, ax in zip(sample_indices, axes.flat):
    visualize_sample(all_records[idx], ax)

plt.tight_layout()
plt.suptitle('Примеры изображений с аннотациями', fontsize=16, y=1.02)
plt.show()

## 6. Анализ качества: пропуски и проблемы

In [ ]:
issues = {
    'no_annotations': 0,
    'single_point_polygons': 0,
    'empty_class_names': 0,
    'missing_images': 0,
    'zero_area_masks': 0,
}

for record in all_records:
    if len(record['annotations']) == 0:
        issues['no_annotations'] += 1
    
    img_path = record.get('image_path', '')
    if img_path and not Path(img_path).exists():
        issues['missing_images'] += 1
    
    for ann in record['annotations']:
        if len(ann['polygon']) < 6:
            issues['single_point_polygons'] += 1
        if not ann.get('category_name', '').strip():
            issues['empty_class_names'] += 1
        mask = polygon_to_mask(record['height'], record['width'], ann['polygon'])
        if mask.sum() == 0:
            issues['zero_area_masks'] += 1

print('Проверка качества данных:')
for issue_name, count in issues.items():
    status = '✓' if count == 0 else '⚠'
    print(f'  {status} {issue_name}: {count}')

## 7. Итоги EDA

### Ключевые выводы:
1. **Дисбаланс классов:** Деталей существенно больше, чем повреждений. Рекомендуется использовать взвешенный loss или аугментации редких классов.
2. **Вариативность размеров:** Изображения разного разрешения — нужна предобработка (resize/паддинг).
3. **Мелкие объекты:** Значительная доля масок <100 px — модель должна хорошо работать с мелкими объектами, что является вызовом для Mask R-CNN.
4. **Качество аннотаций:** Полигоны детальные, пропусков и ошибок выявлено мало.
5. **Train/Val распределение:** 80/20 сплит выполнен корректно, стратификация по классам не применялась.

### Рекомендации:
- Использовать multiscale training для улучшения работы с разными размерами объектов
- Рассмотреть class-aware sampling для борьбы с дисбалансом
- Добавить аугментации масштабирования (Scale, RandomResizedCrop)
- Мониторить per-class метрики, а не только средние